# Optiver frozen model: Kaggle runtime check

Loads the frozen research model (XGBoost x04_slow + cross-sectional centring) from a private dataset and runs it through the competition's `optiver2023` API on the example test batches. Checks library versions, row-ID alignment, finiteness and parity with the Mac predictions. **This notebook does not submit to the competition.**

In [ ]:
# Runtime-compatibility check for the frozen Optiver model. Does NOT submit.
import sys, glob, json, time, platform, os
import numpy as np, pandas as pd
man = glob.glob('/kaggle/input/**/MANIFEST.json', recursive=True)[0]
INP = os.path.dirname(man)
sys.path.insert(0, INP + '/code')
import sklearn, xgboost
print('python', platform.python_version(), 'numpy', np.__version__, 'pandas', pd.__version__,
      'sklearn', sklearn.__version__, 'xgboost', xgboost.__version__)
from optiver_close.portable import PortableModel
from optiver_close.inference import run_kaggle, StreamingPredictor
report = {'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__,
          'sklearn': sklearn.__version__, 'xgboost': xgboost.__version__}
try:
    import joblib
    full = joblib.load(INP + '/model/model.joblib'); report['joblib_load'] = 'ok'
except Exception as e:
    full = None; report['joblib_load'] = f'failed: {type(e).__name__}: {e}'[:300]
portable = PortableModel(INP + '/model').artifact()

class Recorder:
    def __init__(self, env): self.env, self.tests, self.calls = env, [], []
    def iter_test(self):
        for item in self.env.iter_test():
            self.tests.append(item[0].copy()); yield item
    def predict(self, frame):
        self.calls.append(frame.copy()); self.env.predict(frame)

COMP = '/kaggle/input/competitions/optiver-trading-at-the-close'
print('competition files:', sorted(os.listdir(COMP)), sorted(os.listdir(COMP + '/optiver2023')))

class LocalEnv:
    """Same protocol as optiver2023 (one (test, revealed_targets, sample) batch per time_id;
    predict exactly once per batch), served from the competition's example_test_files."""
    def __init__(self, folder):
        self.test = pd.read_csv(folder + '/test.csv')
        self.revealed = pd.read_csv(folder + '/revealed_targets.csv')
        self.sample = pd.read_csv(folder + '/sample_submission.csv')
        self.status = 'ready'
    def iter_test(self):
        for time_id in self.test.time_id.drop_duplicates():
            self.status = 'prediction_needed'
            yield (self.test[self.test.time_id == time_id].reset_index(drop=True),
                   self.revealed[self.revealed.time_id == time_id].reset_index(drop=True),
                   self.sample[self.sample.time_id == time_id].reset_index(drop=True))
            if self.status != 'prediction_received':
                raise RuntimeError('predict() not called exactly once')
    def predict(self, frame):
        if self.status != 'prediction_needed':
            raise RuntimeError('predict() called twice or out of order')
        self.status = 'prediction_received'

try:
    import optiver2023
    base_env = optiver2023.make_env(); report['api'] = 'optiver2023 (official)'
except Exception as e:
    report['api'] = f'local protocol re-implementation on {COMP}/example_test_files; official optiver2023 failed: {type(e).__name__}: {e}'
    base_env = LocalEnv(COMP + '/example_test_files')
env = Recorder(base_env)
t = time.time()
run_kaggle(portable, env)
report['seconds'] = round(time.time() - t, 1)
sub = pd.concat(env.calls)
ref = pd.read_csv(INP + '/model/local_reference_predictions.csv').set_index('row_id').target
report.update(batches=len(env.calls), rows=len(sub), unique_row_ids=int(sub.row_id.nunique()),
              finite=bool(np.isfinite(sub.target).all()),
              row_ids_match_local=bool(set(sub.row_id) == set(ref.index)))
if report['row_ids_match_local']:
    report['portable_vs_local_mac_max_abs_diff'] = float(np.abs(sub.set_index('row_id').target - ref.reindex(sub.row_id)).max())
if full is not None:
    sp = StreamingPredictor(full)
    p = pd.concat([pd.Series(sp.predict(x), index=x.row_id) for x in env.tests])
    report['joblib_vs_portable_max_abs_diff'] = float(np.abs(p - sub.set_index('row_id').target.reindex(p.index)).max())
print(json.dumps(report, indent=1))
